# LWSO-YOLO11n — Train trên Kaggle (VisDrone2019)

Notebook **tự chứa toàn bộ code**: các cell `%%writefile` tái tạo package `lwso`
(SPDConv, C3k2Ghost, EMA, DySample, BiFPNCat + loss CIoU⊕NWD) và các file YAML kiến trúc.

## Chuẩn bị (làm 1 lần)
1. Nén thư mục local `datasets/VisDrone` (giữ nguyên cấu trúc `images/{train,val,test}` + `labels/{train,val,test}`) thành zip.
2. Kaggle → **Datasets → New Dataset** → upload zip (Kaggle tự giải nén).
3. Trong notebook này: **Add Input** → chọn dataset vừa tạo.
4. **Settings → Accelerator → GPU P100** (khuyên dùng P100 1-GPU; code custom module chưa hỗ trợ DDP nên không dùng T4 x2).

## Thời gian
~6-8 phút/epoch @960 trên P100 → `EPOCHS=100` vừa khít phiên 12h.
Muốn train dài hơn: sau mỗi phiên **Save Version** để giữ `runs/`, phiên sau đặt `RESUME_FROM` trỏ tới `last.pt` (attach output của version trước làm Input).


In [ ]:
# ===================== CẤU HÌNH =====================
MODEL_CFG    = "cfg/lwso-yolo11n.yaml"  # model đề xuất
# MODEL_CFG  = "cfg/ablation/yolo11n-p2-nop5.yaml"  # ablation: chỉ +P2/-P5
# MODEL_CFG  = "yolo11n.pt"                          # baseline COCO-pretrained

IMGSZ        = 960
EPOCHS       = 100
BATCH        = 8          # P100 16GB: 8 an toàn, thử 12 nếu còn VRAM
RUN_NAME     = "lwso-n-960"
USE_NWD      = True       # False khi chạy baseline để so sánh công bằng
NWD_RATIO    = 0.5        # trọng số CIoU trong blend; (1-ratio) cho NWD
NWD_CONSTANT = 12.8       # nên ablate {2, 4, 8, 12.8}
RESUME_FROM  = ""         # vd "/kaggle/input/<output-version-truoc>/runs/detect/lwso-n-960/weights/last.pt"


In [ ]:
%pip install -q "ultralytics>=8.3,<8.4"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
!mkdir -p lwso cfg/ablation


In [ ]:
%%writefile lwso/modules.py
"""Custom building blocks for LWSO-YOLO (LightWeight Small-Object YOLO).

Modules:
    SPDConv   - space-to-depth downsampling (no information loss, SPD-Conv paper).
    C3k2Ghost - C3k2 block with GhostBottleneck inner blocks (~40% fewer FLOPs).
    EMA       - Efficient Multi-scale Attention (Ouyang et al., ICASSP 2023).
    DySample  - dynamic learnable upsampler, 'lp' style (Liu et al., ICCV 2023).
    BiFPNCat  - weighted feature concat (BiFPN fast normalized fusion), drop-in
                replacement for Concat in YAML.

All channel-changing modules follow the ultralytics (c1, c2, *args) convention so
they can be registered into parse_model's `base_modules` set (see register.py).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.nn.modules.block import C2f, GhostBottleneck
from ultralytics.nn.modules.conv import Conv

__all__ = ["SPDConv", "C3k2Ghost", "EMA", "DySample", "BiFPNCat"]


class SPDConv(nn.Module):
    """Space-to-depth downsample: each 2x2 spatial patch is rearranged into 4 channels,
    then fused by a stride-1 conv. Halves H/W like a stride-2 conv but discards no pixels,
    which matters for objects only a few pixels wide.
    """

    def __init__(self, c1: int, c2: int, k: int = 3):
        super().__init__()
        self.conv = Conv(c1 * 4, c2, k, 1)

    def forward(self, x):
        if x.shape[-1] % 2 or x.shape[-2] % 2:  # pad odd H/W to even so the 2x2 slices align
            x = F.pad(x, (0, x.shape[-1] % 2, 0, x.shape[-2] % 2))
        x = torch.cat(
            [x[..., ::2, ::2], x[..., 1::2, ::2], x[..., ::2, 1::2], x[..., 1::2, 1::2]], 1
        )
        return self.conv(x)


class C3k2Ghost(C2f):
    """C3k2 with GhostBottleneck inner blocks.

    Signature mirrors C3k2(c1, c2, n, c3k, e, g, shortcut) so YAML args stay compatible;
    `c3k` is accepted but ignored (Ghost blocks are always used).
    """

    def __init__(self, c1, c2, n=1, c3k=False, e=0.5, g=1, shortcut=True):
        super().__init__(c1, c2, n, shortcut, g, e)
        self.m = nn.ModuleList(GhostBottleneck(self.c, self.c) for _ in range(n))


class EMA(nn.Module):
    """Efficient Multi-scale Attention. Channel-preserving (c2 must equal c1);
    the c2 arg exists only to satisfy the (c1, c2, ...) registration convention.
    """

    def __init__(self, c1: int, c2: int = None, factor: int = 8):
        super().__init__()
        assert c2 is None or c2 == c1, f"EMA requires c1 == c2, got {c1} != {c2}"
        assert c1 % factor == 0, f"EMA: channels {c1} must be divisible by factor {factor}"
        self.groups = factor
        self.softmax = nn.Softmax(-1)
        self.agp = nn.AdaptiveAvgPool2d((1, 1))
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.gn = nn.GroupNorm(c1 // self.groups, c1 // self.groups)
        self.conv1x1 = nn.Conv2d(c1 // self.groups, c1 // self.groups, 1)
        self.conv3x3 = nn.Conv2d(c1 // self.groups, c1 // self.groups, 3, padding=1)

    def forward(self, x):
        b, c, h, w = x.size()
        group_x = x.reshape(b * self.groups, -1, h, w)
        x_h = self.pool_h(group_x)
        x_w = self.pool_w(group_x).permute(0, 1, 3, 2)
        hw = self.conv1x1(torch.cat([x_h, x_w], dim=2))
        x_h, x_w = torch.split(hw, [h, w], dim=2)
        x1 = self.gn(group_x * x_h.sigmoid() * x_w.permute(0, 1, 3, 2).sigmoid())
        x2 = self.conv3x3(group_x)
        x11 = self.softmax(self.agp(x1).reshape(b * self.groups, -1, 1).permute(0, 2, 1))
        x12 = x2.reshape(b * self.groups, c // self.groups, -1)
        x21 = self.softmax(self.agp(x2).reshape(b * self.groups, -1, 1).permute(0, 2, 1))
        x22 = x1.reshape(b * self.groups, c // self.groups, -1)
        weights = (torch.matmul(x11, x12) + torch.matmul(x21, x22)).reshape(
            b * self.groups, 1, h, w
        )
        return (group_x * weights.sigmoid()).reshape(b, c, h, w)


def _normal_init(module: nn.Module, mean: float = 0.0, std: float = 1.0, bias: float = 0.0):
    if hasattr(module, "weight") and module.weight is not None:
        nn.init.normal_(module.weight, mean, std)
    if hasattr(module, "bias") and module.bias is not None:
        nn.init.constant_(module.bias, bias)


class DySample(nn.Module):
    """DySample dynamic upsampler ('lp' style). Channel-preserving; c2 exists only for
    the (c1, c2, ...) registration convention. Near-zero cost replacement for
    nn.Upsample(nearest) with learned sampling offsets.
    """

    def __init__(self, c1: int, c2: int = None, scale: int = 2, groups: int = 4):
        super().__init__()
        assert c2 is None or c2 == c1, f"DySample requires c1 == c2, got {c1} != {c2}"
        assert c1 % groups == 0, f"DySample: channels {c1} must be divisible by groups {groups}"
        self.scale = scale
        self.groups = groups
        self.offset = nn.Conv2d(c1, 2 * groups * scale**2, 1)
        _normal_init(self.offset, std=0.001)
        self.register_buffer("init_pos", self._init_pos())

    def _init_pos(self):
        h = torch.arange((-self.scale + 1) / 2, (self.scale - 1) / 2 + 1) / self.scale
        return (
            torch.stack(torch.meshgrid([h, h], indexing="ij"))
            .transpose(1, 2)
            .repeat(1, self.groups, 1)
            .reshape(1, -1, 1, 1)
        )

    def _sample(self, x, offset):
        b, _, h, w = offset.shape
        offset = offset.view(b, 2, -1, h, w)
        coords_h = torch.arange(h, device=x.device) + 0.5
        coords_w = torch.arange(w, device=x.device) + 0.5
        coords = (
            torch.stack(torch.meshgrid([coords_w, coords_h], indexing="ij"))
            .transpose(1, 2)
            .unsqueeze(1)
            .unsqueeze(0)
            .type(x.dtype)
        )
        normalizer = torch.tensor([w, h], dtype=x.dtype, device=x.device).view(1, 2, 1, 1, 1)
        coords = 2 * (coords + offset) / normalizer - 1
        coords = (
            F.pixel_shuffle(coords.view(b, -1, h, w), self.scale)
            .view(b, 2, -1, self.scale * h, self.scale * w)
            .permute(0, 2, 3, 4, 1)
            .contiguous()
            .flatten(0, 1)
        )
        return F.grid_sample(
            x.reshape(b * self.groups, -1, h, w),
            coords,
            mode="bilinear",
            align_corners=False,
            padding_mode="border",
        ).view(b, -1, self.scale * h, self.scale * w)

    def forward(self, x):
        offset = self.offset(x) * 0.25 + self.init_pos
        return self._sample(x, offset)


class BiFPNCat(nn.Module):
    """Weighted concat of n feature maps (BiFPN fast normalized fusion).

    Drop-in for Concat in model YAML: `[[-1, 5], 1, BiFPNCat, [2]]` where the arg
    is the number of inputs. Output channels = sum of input channels, handled by
    the patched parse_model Concat branch.
    """

    def __init__(self, n: int = 2, dimension: int = 1):
        super().__init__()
        self.d = dimension
        self.w = nn.Parameter(torch.ones(n), requires_grad=True)
        self.eps = 1e-4

    def forward(self, x):
        w = F.relu(self.w)
        w = w / (w.sum() + self.eps)
        return torch.cat([w[i] * xi for i, xi in enumerate(x)], self.d)


In [ ]:
%%writefile lwso/register.py
"""Register LWSO custom modules into ultralytics without forking the repo.

ultralytics' parse_model() resolves module names via its own module globals and
infers channels only for classes listed in two frozensets defined *inside* the
function body. We therefore:

  1. setattr() our classes onto ultralytics.nn.tasks so name lookup succeeds, and
  2. rewrite parse_model's source (4 targeted, verified substitutions) and exec it
     back into the tasks namespace so channel inference covers our modules.

Tested against ultralytics 8.3.x (pinned in requirements.txt). Every substitution
must match exactly once, otherwise a RuntimeError explains which one failed —
this is the canary for an incompatible ultralytics version.
"""

import inspect
import re

import ultralytics.nn.tasks as tasks

from .modules import BiFPNCat, C3k2Ghost, DySample, EMA, SPDConv

_CUSTOM_CLASSES = (SPDConv, C3k2Ghost, EMA, DySample, BiFPNCat)
_registered = False


def register_lwso() -> None:
    """Idempotent. Call once before building any model from an LWSO YAML."""
    global _registered
    if _registered:
        return

    for cls in _CUSTOM_CLASSES:
        setattr(tasks, cls.__name__, cls)

    src = inspect.getsource(tasks.parse_model)
    substitutions = [
        # channel-inferring modules with (c1, c2, ...) signatures
        (
            r"base_modules = frozenset\(\s*\{",
            "base_modules = frozenset(\n        {\n"
            "            SPDConv,\n            C3k2Ghost,\n            EMA,\n            DySample,",
        ),
        # modules whose 3rd arg is the repeat count n
        (
            r"repeat_modules = frozenset\([^\{]*\{",
            "repeat_modules = frozenset(  # modules with 'repeat' arguments\n        {\n"
            "            C3k2Ghost,",
        ),
        # BiFPNCat concatenates like Concat: c2 = sum of input channels
        (
            r"elif m is Concat:",
            "elif m is Concat or m is BiFPNCat:",
        ),
        # C3k2Ghost behaves like C3k2 (non-legacy Detect head, c3k flag on m/l/x)
        (
            r"if m is C3k2:",
            "if m is C3k2 or m is C3k2Ghost:",
        ),
    ]
    for pattern, replacement in substitutions:
        src, count = re.subn(pattern, replacement, src, count=1)
        if count != 1:
            raise RuntimeError(
                f"register_lwso: could not patch parse_model (pattern not found: {pattern!r}). "
                "Your ultralytics version is incompatible; install the range pinned in "
                "requirements.txt (ultralytics>=8.3,<8.4)."
            )

    code = compile(src, tasks.__file__, "exec")
    exec(code, tasks.__dict__)  # rebinds tasks.parse_model to the patched version
    _registered = True


In [ ]:
%%writefile lwso/losses.py
"""Small-object-friendly regression loss for ultralytics YOLO.

Blends CIoU with Normalized Wasserstein Distance (NWD, Wang et al. 2021):

    reg = ratio * (1 - CIoU) + (1 - ratio) * (1 - NWD)

NWD models boxes as 2D Gaussians and stays smooth under the few-pixel offsets
that make plain IoU gradients unstable on tiny objects.

Note on `constant`: the original paper uses C=12.8 for absolute-pixel boxes.
Inside v8DetectionLoss, boxes are in stride-normalized (feature-map) units, so
the effective object sizes are smaller; treat C as a tunable (try 2-16) and
ablate. Default keeps 12.8 as a conservative starting point.
"""

import torch

from ultralytics.utils import loss as uloss
from ultralytics.utils.metrics import bbox_iou
from ultralytics.utils.tal import bbox2dist

__all__ = ["nwd", "NWDBboxLoss", "patch_nwd_loss"]


def nwd(pred: torch.Tensor, target: torch.Tensor, constant: float = 12.8, eps: float = 1e-7):
    """Normalized Wasserstein Distance similarity between xyxy boxes. Returns values in (0, 1]."""
    pcx, pcy = (pred[..., 0] + pred[..., 2]) / 2, (pred[..., 1] + pred[..., 3]) / 2
    tcx, tcy = (target[..., 0] + target[..., 2]) / 2, (target[..., 1] + target[..., 3]) / 2
    pw, ph = pred[..., 2] - pred[..., 0], pred[..., 3] - pred[..., 1]
    tw, th = target[..., 2] - target[..., 0], target[..., 3] - target[..., 1]
    dist2 = (pcx - tcx) ** 2 + (pcy - tcy) ** 2 + ((pw - tw) ** 2 + (ph - th) ** 2) / 4
    return torch.exp(-torch.sqrt(dist2.clamp(min=eps)) / constant)


class NWDBboxLoss(uloss.BboxLoss):
    """BboxLoss with the CIoU/NWD blend above; DFL part is unchanged from the parent."""

    nwd_ratio = 0.5  # weight of the CIoU term; (1 - ratio) goes to NWD
    nwd_constant = 12.8

    def forward(
        self,
        pred_dist,
        pred_bboxes,
        anchor_points,
        target_bboxes,
        target_scores,
        target_scores_sum,
        fg_mask,
    ):
        weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1)
        iou = bbox_iou(pred_bboxes[fg_mask], target_bboxes[fg_mask], xywh=False, CIoU=True)
        nwd_sim = nwd(
            pred_bboxes[fg_mask], target_bboxes[fg_mask], constant=self.nwd_constant
        ).unsqueeze(-1)
        reg = self.nwd_ratio * (1.0 - iou) + (1.0 - self.nwd_ratio) * (1.0 - nwd_sim)
        loss_iou = (reg * weight).sum() / target_scores_sum

        if self.dfl_loss:
            target_ltrb = bbox2dist(anchor_points, target_bboxes, self.dfl_loss.reg_max - 1)
            loss_dfl = (
                self.dfl_loss(
                    pred_dist[fg_mask].view(-1, self.dfl_loss.reg_max), target_ltrb[fg_mask]
                )
                * weight
            )
            loss_dfl = loss_dfl.sum() / target_scores_sum
        else:
            loss_dfl = torch.tensor(0.0).to(pred_dist.device)

        return loss_iou, loss_dfl


def patch_nwd_loss(ratio: float = 0.5, constant: float = 12.8) -> None:
    """Swap ultralytics' BboxLoss for the NWD blend. Call before model.train().

    v8DetectionLoss instantiates `BboxLoss` from the loss module's globals, so the
    class swap takes effect for every detection trainer created afterwards.
    """
    NWDBboxLoss.nwd_ratio = float(ratio)
    NWDBboxLoss.nwd_constant = float(constant)
    uloss.BboxLoss = NWDBboxLoss


In [ ]:
%%writefile lwso/__init__.py
"""LWSO-YOLO: lightweight small-object YOLO for VisDrone2019, built on ultralytics."""

from .losses import NWDBboxLoss, nwd, patch_nwd_loss
from .modules import BiFPNCat, C3k2Ghost, DySample, EMA, SPDConv
from .register import register_lwso

__all__ = [
    "SPDConv",
    "C3k2Ghost",
    "EMA",
    "DySample",
    "BiFPNCat",
    "register_lwso",
    "patch_nwd_loss",
    "NWDBboxLoss",
    "nwd",
]


In [ ]:
%%writefile cfg/lwso-yolo11n.yaml
# LWSO-YOLO11 — LightWeight Small-Object YOLO for VisDrone2019
# Base: YOLO11n. Changes vs stock:
#   * P5 stage removed, P2 (stride-4) detect head added -> detect heads at P2/P3/P4
#   * all stride-2 convs replaced by SPDConv (lossless space-to-depth downsampling)
#   * C3k2 -> C3k2Ghost (GhostBottleneck inner blocks)
#   * EMA attention on P3; C2PSA kept on P4
#   * neck: BiFPN-style weighted fusion (BiFPNCat) + DySample learned upsampling
# Requires lwso.register_lwso() before YOLO(<this file>).
# Scale is taken from the filename suffix (11n / 11s); the -s copy is the KD teacher.

nc: 10 # VisDrone classes

scales: # [depth, width, max_channels]
  n: [1.00, 1.00, 1024] # channels below are written for the n scale
  s: [1.33, 1.50, 1024] # teacher for knowledge distillation

backbone:
  # [from, repeats, module, args]                    (shapes @ 960x960 input, n scale)
  - [-1, 1, Conv, [16, 3, 2]] # 0  P1/2   16x480x480
  - [-1, 1, SPDConv, [32]] # 1  P2/4   32x240x240
  - [-1, 2, C3k2Ghost, [64, False, 0.25]] # 2         64x240x240
  - [-1, 1, SPDConv, [64]] # 3  P3/8   64x120x120
  - [-1, 2, C3k2Ghost, [128, False, 0.25]] # 4        128x120x120
  - [-1, 1, EMA, [128]] # 5         128x120x120
  - [-1, 1, SPDConv, [128]] # 6  P4/16  128x60x60
  - [-1, 2, C3k2Ghost, [128, True]] # 7               128x60x60
  - [-1, 1, SPPF, [128, 5]] # 8               128x60x60
  - [-1, 1, C2PSA, [128]] # 9               128x60x60

head:
  # top-down
  - [-1, 1, DySample, [128, 2]] # 10 P4 -> P3 size
  - [[-1, 5], 1, BiFPNCat, [2]] # 11 128+128=256
  - [-1, 1, C3k2Ghost, [128, False]] # 12 P3 intermediate
  - [-1, 1, DySample, [128, 2]] # 13 P3 -> P2 size
  - [[-1, 2], 1, BiFPNCat, [2]] # 14 128+64=192
  - [-1, 1, C3k2Ghost, [64, False]] # 15 P2/4 out (64x240x240)
  # bottom-up
  - [-1, 1, SPDConv, [128]] # 16 P2 -> P3 size
  - [[-1, 12], 1, BiFPNCat, [2]] # 17 128+128=256
  - [-1, 1, C3k2Ghost, [128, False]] # 18 P3/8 out
  - [-1, 1, SPDConv, [128]] # 19 P3 -> P4 size
  - [[-1, 9], 1, BiFPNCat, [2]] # 20 128+128=256
  - [-1, 1, C3k2Ghost, [128, True]] # 21 P4/16 out

  - [[15, 18, 21], 1, Detect, [nc]] # 22 Detect(P2, P3, P4)


In [ ]:
%%writefile cfg/ablation/yolo11n-p2-nop5.yaml
# Ablation step 1: stock YOLO11n with P2 head added and P5 stage removed.
# Uses only stock ultralytics modules -> runs WITHOUT register_lwso().
# Isolates the effect of "+P2 / -P5" before any custom module is introduced.

nc: 10 # VisDrone classes

scales: # [depth, width, max_channels]
  n: [1.00, 1.00, 1024]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [16, 3, 2]] # 0  P1/2
  - [-1, 1, Conv, [32, 3, 2]] # 1  P2/4
  - [-1, 2, C3k2, [64, False, 0.25]] # 2
  - [-1, 1, Conv, [64, 3, 2]] # 3  P3/8
  - [-1, 2, C3k2, [128, False, 0.25]] # 4
  - [-1, 1, Conv, [128, 3, 2]] # 5  P4/16
  - [-1, 2, C3k2, [128, True]] # 6
  - [-1, 1, SPPF, [128, 5]] # 7
  - [-1, 1, C2PSA, [128]] # 8

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 9
  - [[-1, 4], 1, Concat, [1]] # 10 128+128=256
  - [-1, 2, C3k2, [128, False]] # 11 P3 intermediate
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 12
  - [[-1, 2], 1, Concat, [1]] # 13 128+64=192
  - [-1, 2, C3k2, [64, False]] # 14 P2/4 out
  - [-1, 1, Conv, [64, 3, 2]] # 15
  - [[-1, 11], 1, Concat, [1]] # 16 64+128=192
  - [-1, 2, C3k2, [128, False]] # 17 P3/8 out
  - [-1, 1, Conv, [128, 3, 2]] # 18
  - [[-1, 8], 1, Concat, [1]] # 19 128+128=256
  - [-1, 2, C3k2, [128, True]] # 20 P4/16 out

  - [[14, 17, 20], 1, Detect, [nc]] # 21 Detect(P2, P3, P4)


In [ ]:
# ============ Tìm dataset trong /kaggle/input và tạo data yaml ============
import glob
from pathlib import Path

patterns = ["/kaggle/input/*/VisDrone/images/train",
            "/kaggle/input/*/images/train",
            "/kaggle/input/*/*/images/train"]
hits = [p for pat in patterns for p in glob.glob(pat)]
assert hits, "Khong tim thay images/train duoi /kaggle/input — kiem tra lai dataset da attach chua"
DATA_ROOT = str(Path(sorted(set(hits))[0]).parents[1])
print("Dataset root:", DATA_ROOT)

names = ["pedestrian", "people", "bicycle", "car", "van",
         "truck", "tricycle", "awning-tricycle", "bus", "motor"]
yaml_text = (
    f"path: {DATA_ROOT}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(names))
)
Path("visdrone.yaml").write_text(yaml_text)
print(yaml_text)


In [ ]:
# ============ Sanity check: build model, dem params/GFLOPs ============
from lwso import register_lwso
register_lwso()
from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops, get_num_params

m = YOLO(MODEL_CFG).model
print(f"{MODEL_CFG}: {get_num_params(m)/1e6:.2f}M params, "
      f"{get_flops(m, IMGSZ):.1f} GFLOPs@{IMGSZ}, strides {sorted(int(s) for s in m.stride.tolist())}")
del m


In [ ]:
# ===================== TRAIN =====================
from lwso import register_lwso, patch_nwd_loss
register_lwso()
if USE_NWD:
    patch_nwd_loss(ratio=NWD_RATIO, constant=NWD_CONSTANT)
    print(f"[lwso] NWD blend loss: ratio={NWD_RATIO}, C={NWD_CONSTANT}")

from ultralytics import YOLO

model = YOLO(RESUME_FROM if RESUME_FROM else MODEL_CFG)
model.train(
    data="visdrone.yaml",
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    device=0,          # 1 GPU — custom module chua ho tro DDP spawn cua ultralytics
    name=RUN_NAME,
    cos_lr=True,
    close_mosaic=15,
    mixup=0.1,
    patience=50,
    workers=2,
    plots=True,
)


In [ ]:
# ===================== VALIDATE + XUAT KET QUA =====================
import shutil
from lwso import register_lwso
register_lwso()
from ultralytics import YOLO

best = f"runs/detect/{RUN_NAME}/weights/best.pt"
model = YOLO(best)
metrics = model.val(data="visdrone.yaml", imgsz=IMGSZ, split="val")
print(f"\nmAP50: {metrics.box.map50:.4f} | mAP50-95: {metrics.box.map:.4f}")
for name, ap in zip(metrics.names.values(), metrics.box.maps):
    print(f"  {name:>16}: {ap:.4f}")

shutil.copy(best, f"/kaggle/working/best_{RUN_NAME}.pt")
print(f"\nDa copy best.pt -> /kaggle/working/best_{RUN_NAME}.pt (tai ve o tab Output)")


## Sau khi train

- **Save Version** (Save & Run All hoặc Quick Save) để giữ `runs/` + `best_*.pt` trong Output.
- Train tiếp phiên sau: attach Output của version trước làm Input, đặt
  `RESUME_FROM = "/kaggle/input/<slug>/runs/detect/lwso-n-960/weights/last.pt"`.
- Chạy ablation: đổi `MODEL_CFG` sang `cfg/ablation/yolo11n-p2-nop5.yaml` (đặt `USE_NWD=False`),
  và baseline `yolo11n.pt` — so sánh 3 số mAP50 để có bảng ablation.
- Đánh giá test-dev: đổi `split="test"` ở cell validate.
